<!--
---
PURPOSE: Perform eye tracking QC and derive features.
REQUIRES:
  - outputs/reports/session_inventory.parquet
PRODUCES:
  - outputs/eye/session_{id}_eye_features.parquet
WHAT_NEXT: notebooks/05_Video_IO_and_Frame_Timebase.ipynb
---
-->

# 04 Eye Tracking QC and Features

**Purpose**
Perform eye tracking QC and derive features.

**Requires**
- `outputs/reports/session_inventory.parquet`

**Produces**
- `outputs/eye/session_{id}_eye_features.parquet`

**What to run next**
- `notebooks/05_Video_IO_and_Frame_Timebase.ipynb`

We load eye data (if present), run QC, and export features.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "04_Eye_Tracking_QC_and_Features.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Load eye features
If eye tracking is missing, we record a flag and continue.

In [ ]:
from io_sessions import load_sessions_csv, get_session_bundle
from qc import eye_qc_summary
from viz import plot_eye_qc

sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    eye = bundle.load_eye_features()
    print(session_id, eye_qc_summary(eye))
    plot_eye_qc(eye)